# Notebook 14: Clustering – K-Means, DBSCAN, Hierarchical & Gaussian Mixtures
### Part 14/30 – ML Mastery Series for Python Experts

## Clustering – Finding Structure Without Labels

- **Unsupervised learning** discovers hidden groupings without target labels to guide optimization
- No ground truth means evaluation relies on internal metrics and domain validation rather than accuracy
- Each algorithm makes different assumptions: K-Means assumes spherical clusters, DBSCAN assumes density connectivity, GMM assumes Gaussian distributions
- **Use cases**: Customer segmentation for marketing, anomaly detection in fraud/security, image compression via color quantization, preprocessing step for supervised learning
- Distance-based methods are extremely sensitive to feature scaling — always standardize before clustering
- Hyperparameter selection is challenging without labels; requires domain knowledge and multiple metrics
- Visualization is essential: cluster quality is often obvious visually but hidden in summary statistics
- The "right" number of clusters is often subjective and problem-dependent rather than statistically pure

## Learning Objectives

By the end of this notebook, you will be able to:

- Understand the K-Means objective function and Lloyd's iterative optimization algorithm
- Choose optimal k using elbow method and silhouette analysis with visual diagnostics
- Apply DBSCAN to discover arbitrarily-shaped clusters and identify noise points
- Interpret hierarchical clustering dendrograms and cut trees at appropriate linkage distances
- Fit Gaussian Mixture Models and interpret probabilistic cluster responsibilities (soft assignments)
- Compare algorithm behavior on identical datasets to understand their assumptions
- Evaluate clustering quality using internal metrics: silhouette, Davies-Bouldin, Calinski-Harabasz
- Preprocess data appropriately (scaling, dimensionality reduction) for clustering algorithms
- Recognize failure modes: K-Means on non-convex data, DBSCAN with wrong density parameters
- Apply clustering to real datasets and validate against known structure when available

## 🔴 1. K-Means – Centroid-Based Clustering

You already know supervised models — now let's discover hidden groups in data without any teacher telling us the answers. K-Means partitions data into k spherical clusters by minimizing within-cluster sum of squares (WCSS). The algorithm iterates between assigning points to nearest centroids and recomputing centroids until convergence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris, load_wine, load_digits, fetch_california_housing
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.metrics import silhouette_samples
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style="whitegrid")

# Generate synthetic blob data with 4 clear clusters
X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=42)
print(f"Data shape: {X_blobs.shape}")
print(f"True cluster distribution: {np.bincount(y_true)}")

In [ ]:
# Fit K-Means with k=4 (known from data generation)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X_blobs)
centroids = kmeans.cluster_centers_

# Visualize clusters and centroids
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_kmeans, cmap='viridis', s=50, alpha=0.7)
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200, linewidths=3, label='Centroids')
plt.title('K-Means Clustering (k=4)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.colorbar(scatter, label='Cluster')
plt.show()

print(f"Inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"Number of iterations to converge: {kmeans.n_iter_}")

In [ ]:
# Show how inertia decreases with more clusters (diminishing returns)
k_range = range(1, 11)
inertias = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertias, 'bo-', markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (WCSS)')
plt.title('K-Means Inertia vs Number of Clusters')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()

print("Notice the steep drop from k=1 to k=4, then diminishing returns — this hints at the elbow method")

## 📐 2. Choosing Optimal k – Elbow & Silhouette

The elbow method looks for diminishing returns in inertia, while silhouette score measures how similar objects are to their own cluster (cohesion) compared to other clusters (separation). Range is [-1, 1], with higher values indicating better defined clusters.

In [ ]:
# Elbow method visualization
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(k_range, inertias, 'bo-', markersize=8)
plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='True k=4')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.legend()
plt.grid(True, alpha=0.3)

# Silhouette scores
silhouette_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    score = silhouette_score(X_blobs, labels)
    silhouette_scores.append(score)

plt.subplot(1, 2, 2)
plt.plot(range(2, 11), silhouette_scores, 'go-', markersize=8)
plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='True k=4')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best silhouette score: {max(silhouette_scores):.3f} at k={np.argmax(silhouette_scores)+2}")

In [ ]:
# Detailed silhouette plots for specific k values
from sklearn.metrics import silhouette_samples

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
k_values = [2, 3, 4, 6]

for idx, k in enumerate(k_values):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    silhouette_avg = silhouette_score(X_blobs, labels)
    sample_silhouette_values = silhouette_samples(X_blobs, labels)
    
    y_lower = 10
    for i in range(k):
        ith_cluster_silhouette_values = sample_silhouette_values[labels == i]
        ith_cluster_silhouette_values.sort()
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        color = plt.cm.nipy_spectral(float(i) / k)
        axes[idx].fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values,
                                facecolor=color, edgecolor=color, alpha=0.7)
        axes[idx].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    axes[idx].axvline(x=silhouette_avg, color="red", linestyle="--", label=f'Avg: {silhouette_avg:.2f}')
    axes[idx].set_title(f'Silhouette Plot for k={k}')
    axes[idx].set_xlabel('Silhouette Coefficient')
    axes[idx].set_ylabel('Cluster')
    axes[idx].set_xlim([-0.1, 1])
    axes[idx].legend()

plt.tight_layout()
plt.show()

print("Thick silhouette plots indicate well-separated clusters; thin/red values indicate misclassification")

## 🌙 3. DBSCAN – Density-Based Clustering

DBSCAN groups together points that are closely packed together, marking points in low-density regions as outliers. It doesn't require specifying k and can find arbitrarily shaped clusters — crucial for non-convex data where K-Means fails.

In [ ]:
# Generate moon-shaped data (non-convex clusters)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Fit DBSCAN with appropriate eps and min_samples
dbscan = DBSCAN(eps=0.2, min_samples=5)
y_dbscan = dbscan.fit_predict(X_moons)

# Count clusters (excluding noise)
n_clusters = len(set(y_dbscan)) - (1 if -1 in y_dbscan else 0)
n_noise = list(y_dbscan).count(-1)
print(f"Estimated clusters: {n_clusters}")
print(f"Noise points: {n_noise} ({n_noise/len(y_dbscan)*100:.1f}%)")

In [ ]:
# Visualize DBSCAN results
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.scatter(X_moons[:, 0], X_moons[:, 1], c='gray', alpha=0.5)
plt.title('Original Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 3, 2)
# Color by cluster, noise in black
unique_labels = set(y_dbscan)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
for label, color in zip(unique_labels, colors):
    if label == -1:
        color = [0, 0, 0, 1]  # Black for noise
    mask = y_dbscan == label
    plt.scatter(X_moons[mask, 0], X_moons[mask, 1], c=[color], s=50, label=f'Cluster {label}' if label != -1 else 'Noise')
plt.title(f'DBSCAN (eps=0.2, min=5)\nClusters: {n_clusters}, Noise: {n_noise}')
plt.xlabel('Feature 1')
plt.legend()

# Compare with K-Means on same data
plt.subplot(1, 3, 3)
kmeans_moons = KMeans(n_clusters=2, random_state=42)
y_kmeans_moons = kmeans_moons.fit_predict(X_moons)
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_kmeans_moons, cmap='viridis', s=50)
plt.title('K-Means (k=2) on Moons\n(Fails on non-convex shape)')
plt.xlabel('Feature 1')

plt.tight_layout()
plt.show()

print("K-Means splits the moons incorrectly; DBSCAN correctly identifies the crescent shapes")

In [ ]:
# Demonstrate sensitivity to eps parameter
eps_values = [0.1, 0.2, 0.3, 0.5]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, eps in zip(axes, eps_values):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    unique_labels = set(labels)
    for label in unique_labels:
        mask = labels == label
        color = 'black' if label == -1 else plt.cm.Spectral(label / max(unique_labels))
        ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c=[color], s=30, alpha=0.7)
    ax.set_title(f'eps={eps}\nClusters: {n_clusters}, Noise: {n_noise}')
    ax.set_xlabel('Feature 1')

axes[0].set_ylabel('Feature 2')
plt.suptitle('DBSCAN Sensitivity to eps Parameter', y=1.02)
plt.tight_layout()
plt.show()

print("Too small eps → everything is noise; too large eps → everything merges into one cluster")

## 🌲 4. Agglomerative Hierarchical Clustering & Dendrograms

Hierarchical clustering builds nested clusters by merging or splitting them successively. The dendrogram visualization shows the merge history and allows selecting clusters by cutting the tree at a specific linkage distance.

In [ ]:
# Load and scale iris data for hierarchical clustering
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

# Fit Agglomerative Clustering with different linkages
linkages = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, linkage in enumerate(linkages):
    agg = AgglomerativeClustering(n_clusters=3, linkage=linkage)
    y_agg = agg.fit_predict(X_iris)
    
    # Project to 2D for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_iris)
    
    scatter = axes[idx].scatter(X_pca[:, 0], X_pca[:, 1], c=y_agg, cmap='viridis', s=50)
    axes[idx].set_title(f'Agglomerative ({linkage} linkage)')
    axes[idx].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    axes[idx].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')

plt.tight_layout()
plt.show()

print("Ward minimizes variance, complete uses max distance, average uses mean, single can create chains")

In [ ]:
# Create dendrogram using scipy
from scipy.cluster.hierarchy import dendrogram, linkage

# Sample subset for clearer visualization
np.random.seed(42)
sample_idx = np.random.choice(len(X_iris), size=50, replace=False)
X_sample = X_iris[sample_idx]
y_sample = y_iris[sample_idx]

# Compute linkage matrix
linked = linkage(X_sample, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked, orientation='top', labels=y_sample, leaf_rotation=90, leaf_font_size=8)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.xlabel('Sample Index (colored by true species)')
plt.ylabel('Distance')
plt.axhline(y=10, color='r', linestyle='--', label='Cut line (3 clusters)')
plt.legend()
plt.tight_layout()
plt.show()

print("Cut the dendrogram horizontally to obtain clusters; lower cuts yield more clusters")

## 🎲 5. Gaussian Mixture Models – Probabilistic Soft Clustering

GMM assumes data comes from a mixture of Gaussian distributions. Unlike K-Means hard assignments, GMM provides probabilistic (soft) cluster memberships — useful when points genuinely belong to multiple groups or cluster boundaries are fuzzy.

In [ ]:
# Use blobs with different variances to show soft assignment value
X_varied, y_varied = make_blobs(n_samples=400, centers=[[0, 0], [3, 3], [0, 4]], 
                                cluster_std=[0.4, 0.8, 1.2], random_state=42)

# Fit GMM
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X_varied)
y_gmm = gmm.predict(X_varied)
probs = gmm.predict_proba(X_varied)

print(f"GMM converged: {gmm.converged_}")
print(f"Log-likelihood: {gmm.lower_bound_:.2f}")
print(f"AIC: {gmm.aic(X_varied):.2f}, BIC: {gmm.bic(X_varied):.2f}")

In [ ]:
# Visualize GMM soft assignments vs K-Means hard assignments
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# K-Means hard assignment
kmeans_varied = KMeans(n_clusters=3, random_state=42)
y_km_varied = kmeans_varied.fit_predict(X_varied)
axes[0].scatter(X_varied[:, 0], X_varied[:, 1], c=y_km_varied, cmap='viridis', s=50)
axes[0].scatter(kmeans_varied.cluster_centers_[:, 0], kmeans_varied.cluster_centers_[:, 1], 
                c='red', marker='x', s=200, linewidths=3)
axes[0].set_title('K-Means (Hard Assignment)')

# GMM hard assignment (most likely)
axes[1].scatter(X_varied[:, 0], X_varied[:, 1], c=y_gmm, cmap='viridis', s=50)
axes[1].scatter(gmm.means_[:, 0], gmm.means_[:, 1], 
                c='red', marker='x', s=200, linewidths=3)
axes[1].set_title('GMM (Most Likely Assignment)')

# GMM uncertainty (max probability as alpha)
max_probs = probs.max(axis=1)
scatter = axes[2].scatter(X_varied[:, 0], X_varied[:, 1], c=y_gmm, 
                         cmap='viridis', s=50, alpha=max_probs)
axes[2].set_title('GMM Soft Assignment\n(Transparency = Confidence)')
plt.colorbar(scatter, ax=axes[2], label='Max Probability')

plt.tight_layout()
plt.show()

print("Note how GMM captures the elliptical shape and varying densities better than K-Means")

In [ ]:
# Show probability distribution for ambiguous points
uncertain_mask = (probs.max(axis=1) < 0.8) & (probs.max(axis=1) > 0.4)
print(f"Number of uncertain points (40-80% confidence): {uncertain_mask.sum()}")
print("\nSample probabilities for uncertain points:")
print(probs[uncertain_mask][:5].round(3))

## 📊 6. Internal Evaluation Metrics Comparison

Without ground truth, we rely on internal metrics. Silhouette measures separation vs cohesion, Davies-Bouldin measures cluster compactness vs separation (lower is better), and Calinski-Harabasz measures between vs within cluster dispersion.

In [ ]:
# Compare all algorithms on standardized iris data
X_scaled = StandardScaler().fit_transform(iris.data)
algorithms = {
    'K-Means': KMeans(n_clusters=3, random_state=42, n_init=10),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),
    'Agglomerative': AgglomerativeClustering(n_clusters=3, linkage='ward'),
    'GMM': GaussianMixture(n_components=3, random_state=42)
)

results = []
for name, algorithm in algorithms.items():
    if name == 'GMM':
        labels = algorithm.fit_predict(X_scaled)
    else:
        labels = algorithm.fit_predict(X_scaled)
    
    # Skip if DBSCAN found only 1 cluster or all noise
    n_labels = len(set(labels)) - (1 if -1 in labels else 0)
    if n_labels < 2:
        continue
    
    # Calculate metrics (exclude noise for DBSCAN)
    mask = labels != -1 if -1 in labels else slice(None)
    sil = silhouette_score(X_scaled[mask], labels[mask])
    db = davies_bouldin_score(X_scaled[mask], labels[mask])
    ch = calinski_harabasz_score(X_scaled[mask], labels[mask])
    
    # ARI vs true labels (for reference, not available in real unsupervised)
    ari = adjusted_rand_score(y_iris[mask], labels[mask])
    
    results.append({
        'Algorithm': name,
        'Clusters': n_labels,
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch,
        'ARI (vs truth)': ari
    })

results_df = pd.DataFrame(results)
print("Clustering Evaluation Metrics on Iris Dataset:")
print(results_df.round(3))

In [ ]:
# Visualize metric comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']
colors = ['green' if m == 'Silhouette' else 'red' if m == 'Davies-Bouldin' else 'blue' for m in metrics]

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    bars = axes[idx].bar(results_df['Algorithm'], results_df[metric], color=color, alpha=0.7)
    axes[idx].set_title(f'{metric} Score')
    axes[idx].set_ylabel('Score')
    axes[idx].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{height:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nNote: Higher Silhouette/CH is better; lower DB is better. ARI shows alignment with true labels.")

## 🍷 7. Real Dataset Example – Wine Clustering

Let's apply all algorithms to the wine dataset without using the labels, then check how well the discovered clusters align with the actual wine classes as a sanity check.

In [ ]:
# Load and preprocess wine data
wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)
y_wine = wine.target
print(f"Wine dataset: {X_wine.shape[0]} samples, {X_wine.shape[1]} features, {len(np.unique(y_wine))} classes")

# PCA for 2D visualization
pca_wine = PCA(n_components=2)
X_wine_pca = pca_wine.fit_transform(X_wine)
print(f"PCA explains {pca_wine.explained_variance_ratio_.sum():.1%} of variance")

In [ ]:
# Apply clustering algorithms with tuned parameters
wine_clusters = {}

# K-Means
km_wine = KMeans(n_clusters=3, random_state=42, n_init=10)
wine_clusters['K-Means'] = km_wine.fit_predict(X_wine)

# DBSCAN (tuned eps)
db_wine = DBSCAN(eps=1.5, min_samples=5)
wine_clusters['DBSCAN'] = db_wine.fit_predict(X_wine)

# Agglomerative
agg_wine = AgglomerativeClustering(n_clusters=3, linkage='ward')
wine_clusters['Agglomerative'] = agg_wine.fit_predict(X_wine)

# GMM
gmm_wine = GaussianMixture(n_components=3, random_state=42)
wine_clusters['GMM'] = gmm_wine.fit_predict(X_wine)

# Visualize in PCA space
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# True labels
axes[0, 0].scatter(X_wine_pca[:, 0], X_wine_pca[:, 1], c=y_wine, cmap='viridis', s=50)
axes[0, 0].set_title('True Labels (Hidden)')
axes[0, 0].set_xlabel('PC1')
axes[0, 0].set_ylabel('PC2')

# Algorithm results
for idx, (name, labels) in enumerate(wine_clusters.items()):
    row = (idx + 1) // 3
    col = (idx + 1) % 3
    scatter = axes[row, col].scatter(X_wine_pca[:, 0], X_wine_pca[:, 1], 
                                    c=labels, cmap='viridis', s=50)
    ari = adjusted_rand_score(y_wine, labels)
    axes[row, col].set_title(f'{name}\n(ARI: {ari:.3f})')
    axes[row, col].set_xlabel('PC1')
    axes[row, col].set_ylabel('PC2')

# Remove empty subplot
axes[1, 2].remove()
plt.tight_layout()
plt.show()

In [ ]:
# Detailed metrics for wine dataset
wine_results = []
for name, labels in wine_clusters.items():
    mask = labels != -1 if -1 in labels else slice(None)
    sil = silhouette_score(X_wine[mask], labels[mask])
    db = davies_bouldin_score(X_wine[mask], labels[mask])
    ch = calinski_harabasz_score(X_wine[mask], labels[mask])
    ari = adjusted_rand_score(y_wine[mask], labels[mask])
    
    wine_results.append({
        'Algorithm': name,
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch,
        'ARI': ari
    })

wine_results_df = pd.DataFrame(wine_results)
print("Wine Dataset Clustering Results:")
print(wine_results_df.round(3))

print(f"\nBest algorithm by ARI: {wine_results_df.loc[wine_results_df['ARI'].idxmax(), 'Algorithm']}")
print(f"Best algorithm by Silhouette: {wine_results_df.loc[wine_results_df['Silhouette'].idxmax(), 'Algorithm']}")

## ⚠️ Common Pitfalls & Pro Tips

- **Not Scaling Features**: Distance-based methods (K-Means, DBSCAN, Hierarchical) fail on unscaled data — always use StandardScaler
- **K-Means Spherical Assumption**: K-Means will force non-convex clusters into Voronoi cells; use DBSCAN or GMM for irregular shapes
- **Wrong DBSCAN eps**: Too small → everything is noise; too large → single cluster; use k-distance graph to tune
- **Hierarchical Scalability**: O(n²) memory makes it unusable for n > 10,000; use sampling or mini-batch alternatives
- **GMM Initialization**: Random initialization can converge to poor local optima; try multiple init_params
- **Over-relying on Elbow**: The elbow is often ambiguous; always confirm with silhouette or domain knowledge
- **No Single Best Metric**: Silhouette favors convex clusters; use multiple metrics and visual inspection
- **Always Visualize**: High-dimensional clustering can look good on paper but terrible in projection; check PCA/t-SNE plots
- **Cluster in Reduced Dimensions**: For high-d data, PCA before clustering often improves results and speed
- **Ignoring Noise**: DBSCAN's -1 labels are features, not bugs; analyze what makes points outliers
- **Blindly Trusting k**: The "right" k might not exist; consider hierarchical or soft clustering instead
- **Forgetting Random State**: K-Means and GMM have random initialization; set seeds for reproducibility

## 📝 Exercises

### Easy
Run K-Means on the digits dataset (8x8 images) with k=10. Project to 2D using PCA and visualize the clusters colored by their assigned labels. Compare the cluster centers to actual digit images.

### Medium
On make_circles (factor=0.5, noise=0.05), compare K-Means vs DBSCAN cluster boundaries. Show that K-Means fails to separate the concentric circles while DBSCAN succeeds. Visualize the decision boundaries if possible.

### Medium
Use AgglomerativeClustering on the wine dataset with complete linkage. Plot the dendrogram and cut at heights producing 2, 3, and 4 clusters. Compute ARI against true labels for each cut and determine which height produces the best alignment.

### Hard
Implement an automatic k-selection function that combines elbow detection (second derivative) and silhouette maximization. Apply it to make_blobs with varying cluster_std and evaluate how often it correctly identifies the true k.

### Bonus
Apply GMM to the California housing dataset (standardized). Cluster houses into 4 groups and interpret the clusters by examining their centroids in original feature space. Can you identify "expensive urban", "rural", etc.?

<details>
<summary>💡 Exercise Solutions (Click to Expand)</summary>

### Easy Solution: K-Means on Digits
```python
digits = load_digits()
X_dig = StandardScaler().fit_transform(digits.data)

# K-Means
km_digits = KMeans(n_clusters=10, random_state=42)
labels = km_digits.fit_predict(X_dig)

# PCA visualization
pca_dig = PCA(n_components=2)
X_dig_pca = pca_dig.fit_transform(X_dig)
plt.scatter(X_dig_pca[:, 0], X_dig_pca[:, 1], c=labels, cmap='tab10')
plt.title('Digits K-Means Clusters')

# Show cluster centers as images
fig, axes = plt.subplots(2, 5)
for i, ax in enumerate(axes.flat):
    ax.imshow(km_digits.cluster_centers_[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Cluster {i}')
```

### Medium Solution: Circles Comparison
```python
X_circ, y_circ = make_circles(n_samples=300, factor=0.5, noise=0.05, random_state=42)

# K-Means fails
km_circ = KMeans(n_clusters=2).fit_predict(X_circ)
# DBSCAN succeeds
db_circ = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_circ)

fig, axes = plt.subplots(1, 2)
axes[0].scatter(X_circ[:, 0], X_circ[:, 1], c=km_circ)
axes[0].set_title('K-Means (fails)')
axes[1].scatter(X_circ[:, 0], X_circ[:, 1], c=db_circ)
axes[1].set_title('DBSCAN (succeeds)')
```

### Hard Solution: Auto k-Selection
```python
def auto_select_k(X, k_range=range(2, 11)):
    inertias = []
    silhouettes = []
    
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42).fit(X)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X, km.labels_))
    
    # Elbow: find point of maximum curvature (simplified)
    deltas = np.diff(inertias)
    elbow_k = k_range[np.argmax(deltas[:-1] - deltas[1:]) + 1]
    
    # Silhouette: max score
    sil_k = k_range[np.argmax(silhouettes)]
    
    return elbow_k, sil_k, silhouettes
```
</details>

## ✅ Summary – What You Learned Today

- **K-Means**: Fast centroid-based clustering assuming spherical, equally-sized clusters; use elbow and silhouette for k selection
- **DBSCAN**: Density-based clustering that finds arbitrary shapes and identifies noise; sensitive to eps and min_samples parameters
- **Hierarchical**: Agglomerative clustering with dendrogram visualization; linkage choice (ward/complete/average/single) dramatically affects results
- **GMM**: Probabilistic soft clustering providing membership probabilities; better for overlapping clusters and elliptical distributions
- **Evaluation**: No single metric tells the whole story; combine silhouette, Davies-Bouldin, Calinski-Harabasz with visualization
- **Preprocessing**: Always scale features; consider PCA for high-dimensional data
- **Validation**: When ground truth exists (rare), ARI confirms cluster quality; otherwise rely on domain expertise
- **Algorithm Selection**: Match algorithm assumptions to data geometry — no free lunch in clustering

## 🔮 Next Notebook Preview

**Notebook 15: Dimensionality Reduction – PCA, t-SNE, UMAP** — When you have too many features, curse of dimensionality strikes. We'll cover:
- **PCA**: Linear projection maximizing variance, eigenvalue decomposition, explained variance ratios
- **t-SNE**: Non-linear visualization preserving local neighborhoods; perplexity tuning and interpretation
- **UMAP**: Modern manifold learning balancing local and global structure; faster and more stable than t-SNE
- **Feature Extraction**: Using reduced dimensions as input features for downstream models
- **Noise Removal**: Reconstructing data from top k components to filter noise
- **Speeding Up Models**: How dimensionality reduction can improve both training speed and model performance

Get ready to compress high-dimensional chaos into meaningful low-dimensional representations.